# Milestone 1: Data Profiling

Profil awal dataset besar Indonesia menggunakan Polars LazyFrame dan DuckDB. Isi sumber serta deskripsi dataset pada `data/README.md` sebelum menjalankan notebook ini.

In [2]:
from pathlib import Path
import duckdb
import polars as pl

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "stocks.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {DATA_PATH}. "
        "Pastikan stocks.csv sudah diekstrak ke data/raw/."
    )

print(f"Dataset: {DATA_PATH.name}")
print(f"Ukuran file: {DATA_PATH.stat().st_size / 1_000_000:.1f} MB")

Dataset: stocks.csv
Ukuran file: 136.3 MB


## Profil Polars

`scan_parquet` hanya membangun rencana query; file baru dibaca ketika `collect()` dipanggil.

In [3]:
lazy_data = pl.scan_csv(DATA_PATH)
print("Skema:", lazy_data.collect_schema())

ringkasan = lazy_data.select([
    pl.len().alias("jumlah_baris"),
    pl.col("symbol").n_unique().alias("jumlah_emiten"),
    pl.col("datetime").min().alias("waktu_awal"),
    pl.col("datetime").max().alias("waktu_akhir"),
    pl.all().null_count().name.suffix("_null"),
]).collect()

ringkasan

Skema: Schema({'datetime': String, 'symbol': String, 'open': Float64, 'high': Float64, 'low': Float64, 'close': Float64, 'volume': Float64})


jumlah_baris,jumlah_emiten,waktu_awal,waktu_akhir,datetime_null,symbol_null,open_null,high_null,low_null,close_null,volume_null
u32,u32,str,str,u32,u32,u32,u32,u32,u32,u32
2061655,956,"""1990-04-23 02:00:00""","""2025-07-04 02:00:00""",0,0,0,0,0,0,0


In [4]:
contoh_data = lazy_data.head(10).collect()
contoh_data

datetime,symbol,open,high,low,close,volume
str,str,f64,f64,f64,f64,f64
"""2005-06-30 02:00:00""","""IDX:AALI""",3111.92455,3131.620275,3111.92455,3131.620275,1.580926e6
"""2005-07-01 02:00:00""","""IDX:AALI""",3111.92455,3111.92455,3072.5331,3092.228825,901210.0
"""2005-07-04 02:00:00""","""IDX:AALI""",3072.5331,3072.5331,2993.7502,3033.14165,3.226588e6
"""2005-07-05 02:00:00""","""IDX:AALI""",3033.14165,3033.14165,3013.445925,3013.445925,1.928718e6
"""2005-07-06 02:00:00""","""IDX:AALI""",3033.14165,3072.5331,3033.14165,3052.837375,1.632333e6
"""2005-07-07 02:00:00""","""IDX:AALI""",3052.837375,3111.92455,3052.837375,3092.228825,2.272701e6
"""2005-07-08 02:00:00""","""IDX:AALI""",3092.228825,3190.70745,3092.228825,3190.70745,3.253878e6
"""2005-07-11 02:00:00""","""IDX:AALI""",3190.70745,3230.0989,3171.011725,3190.70745,1.20521e6
"""2005-07-12 02:00:00""","""IDX:AALI""",3190.70745,3210.403175,3151.316,3171.011725,1.596793e6


## Profil DuckDB

DuckDB membaca file Parquet langsung dari disk dan menghasilkan ringkasan statistik setiap kolom.

In [7]:
import polars as pl

connection = duckdb.connect()
cursor = connection.execute(
    "SUMMARIZE SELECT * FROM read_csv_auto(?, header=True)",
    [str(DATA_PATH)]
)
columns = [desc[0] for desc in cursor.description]
rows = cursor.fetchall()
profil_duckdb = pl.DataFrame(rows, schema=columns, orient="row")
profil_duckdb

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[38,2]"
"""datetime""","""TIMESTAMP""","""1990-04-23 02:00:00""","""2025-07-04 02:00:00""",8723,"""2018-01-27 02:22:28.601973""",null,"""2014-02-26 08:59:45.354837""","""2019-05-31 08:17:48.840044""","""2022-10-16 15:08:13.208518""",2061655,0.00
"""symbol""","""VARCHAR""","""IDX:AADI""","""IDX:ZYRX""",932,null,null,null,null,null,2061655,0.00
"""open""","""DOUBLE""","""1.0""","""226150.0""",76603,"""1193.5359202012482""","""3601.5994185464074""","""119.63608212392099""","""313.6265051760028""","""910.0675297139846""",2061655,0.00
"""high""","""DOUBLE""","""1.0""","""226150.0""",76074,"""1213.0925827229116""","""3651.322903348834""","""122.84297071976081""","""320.6369877716496""","""927.8540744112613""",2061655,0.00
"""low""","""DOUBLE""","""1.0""","""226150.0""",76985,"""1174.24299314465""","""3550.657823607694""","""117.02907122856162""","""307.35239153284544""","""892.1933128626035""",2061655,0.00
"""close""","""DOUBLE""","""1.0""","""226150.0""",76781,"""1193.8202564397673""","""3599.772669965705""","""119.61125053999675""","""314.4164140851011""","""908.6429599382282""",2061655,0.00
"""volume""","""DOUBLE""","""0.0""","""45941762000.0""",672876,"""20306576.362551607""","""134989236.57559592""","""53314.725235191196""","""942877.8506178042""","""8708510.406942068""",2061655,0.00


In [ ]:
### Ringkasan Temuan Milestone 1
* **Ukuran & jumlah data:** File `stocks.csv` berukuran sekitar **136,3 MB** dengan **2.061.655 baris** dan **7 kolom** (`datetime`, `symbol`, `open`, `high`, `low`, `close`, `volume`).
* **Tipe kolom & nilai null:** Terdiri atas kolom waktu/teks (`datetime`, `symbol`) serta kolom numerik (`open`, `high`, `low`, `close`, `volume`), tanpa nilai *null* awal pada pemeriksaan ringkasan.
* **Anomali yang perlu ditangani di Milestone 2:** Adanya nilai volume minimum `0.0` (`volume == 0`), rentang harga/volume yang lebar antar emiten, serta potensi duplikat baris atau ketidakwajaran nilai OHLC yang perlu diperiksa dan divalidasi saat pembersihan data.
    